In [1]:
pip install --upgrade pyBKT

  Attempting uninstall: pyBKT
    Found existing installation: pyBKT 1.4.2
    Uninstalling pyBKT-1.4.2:
      Successfully uninstalled pyBKT-1.4.2
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install scikit-learn==1.3.0

Note: you may need to restart the kernel to use updated packages.


In [3]:
import random as rand
import matplotlib.pyplot as plt
import itertools
import pandas as pd
from pyBKT.models import Model
import os
import joblib
import json
from ELO import Elo

In [4]:
import numpy as np

In [5]:
with open('../elo_variable.json', 'r') as Elo_Data:
    elo_data = json.load(Elo_Data)

print(elo_data)

{'globals': {'students': 500, 'init_skill_level': [0.0, 0.0], 'k_success': 1, 'k_fail': 0.5}, 'scenarios': [{'id': 1, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 0], [0, 1]], 'difficulty_level': [1, 1], 'depends': [-1, -1]}, {'id': 2, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 1], [1, 1]], 'difficulty_level': [0, 1], 'depends': [-1, -1]}, {'id': 3, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0], [0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1], 'depends': [-1, -1, -1, -1]}, {'id': 4, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1], [0, 1, 1, 1]], 'difficulty_level': [1, 1, 0, 1], 'depends': [-1, -1, -1, -1]}, {'id': 5, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1, 0, 1], [0, 1, 1, 0, 1, 1]], 'difficulty_level': [0, 0, 0, 1, 1, 1], 'depends': [-1, -1, -1, -1, -1, -1]}, {'id': 6, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1, 2, 2], 'depends': [-1, -1, -1, -1, -1, -1]}

In [6]:
def load_training_data(scenario, split):
    base_path = "../Simulated_Data"
    
    filename = f"scenario_{scenario}_{split}_data.csv"
    filepath = os.path.join(base_path, filename)
    
    return pd.read_csv(filepath)

In [7]:
import joblib

def save_bkt(model, scenario_id, num_Skills, base_dir=None):
    # Load the tarin model's parameters
    model_params     = model.params() #
    model_parameters = {} # Initialize the dictionary to store the parameters for each skill
    
    for skill_id in range(num_Skills):
         # Get the parameters for the current skill
        params = model_params.loc[(str(skill_id),), :]
        
        p_start = params.loc[('prior', 'default'), 'value']
        p_trans = params.loc[('learns', 'default'), 'value']
        
        guesses = params.loc[params.index.get_level_values('param').isin(['guesses'])].reset_index(level=0, drop=True)
        p_guesses = guesses.reset_index(level=0)[['class', 'value']]
        
        # Extract slips for all classes for the current skill
        slips = params.loc[params.index.get_level_values('param').isin(['slips'])].reset_index(level=0, drop=True)
        p_slips = slips.reset_index(level=0)[['class', 'value']]
        
        # Store the extracted parameters in the dictionary for future use
        model_parameters[f"p_start_{skill_id}"] = p_start
        model_parameters[f"p_trans_{skill_id}"] = p_trans
        model_parameters[f"p_guesses_{skill_id}"] = p_guesses.to_dict(orient='records')
        model_parameters[f"p_slips_{skill_id}"] = p_slips.to_dict(orient='records')
    
    if base_dir is None:
        base_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'Trained_Models'))

    os.makedirs(base_dir, exist_ok=True)

    model_path = os.path.join(base_dir, f'BKT_model_scenario_{scenario_id}.pth')
    
    # Save the model parameters to a JSON file
    with open(model_path, 'w') as f:
        json.dump(model_parameters, f, indent=4)
    
    # Name the model
    model_filename = f'BKT_model_scenario_{scenario_id}.pkl'    # Filename
    
    # Save the model to the specified path using joblib
    joblib.dump(model, os.path.join(base_dir, model_filename))
     

In [8]:
all_data = {}

level_skill   = [] 
mastery_level = 1.5
i = 0

global_values = elo_data["globals"]
scenarios = elo_data["scenarios"]

students          = global_values["students"]
init_skill_level  = np.array(global_values["init_skill_level"])
k_success         = global_values["k_success"]

In [9]:
defaults = {
    'user_id' :   'student_id',
    'order_id':   'step',           # Assuming 'Task_ID' corresponds to the order ID
    'skill_name': 'skill_id',     # Assuming 'Skill' corresponds to the skill ids
    'correct':    'success',        # Assuming 'Success' corresponds to correct/incorrect values
    'multigs': 'task'
    
}

In [37]:
for scenario in scenarios:
    scenario_id    = scenario["id"]
    num_tasks      = scenario["num_tasks"]
    difficulty_level = np.array(scenario["difficulty_level"])
    q_matrix       = np.array(scenario["q_matrix"])
    num_skills = scenario["num_skills"]
    train_df = load_training_data(scenario_id, "train")
    test_df = load_training_data(scenario_id, "test")

    model = Model(seed = 42, num_fits = 10)
    model.fit(data=train_df, defaults=defaults, multigs = True)#, forgets = True)
    # Perform evalution
    training_acc = model.evaluate(data=train_df, metric='accuracy')
    print(f"Scenario {scenario_id} train Accuracy, {training_acc}")
    print(model.params())
    test_acc = model.evaluate(data=test_df, metric='accuracy')
    print(f"Scenario {scenario_id} test Accuracy, {test_acc}")

    # save_bkt(model, scenario_id, num_skills)

Scenario 1 train Accuracy, 0.6325547796773416
                        value
skill param   class          
0     prior   default 0.08142
      learns  default 0.36063
      guesses 0       0.22025
      slips   0       0.27028
      forgets default 0.00000
1     prior   default 0.01694
      learns  default 0.25338
      guesses 1       0.27037
      slips   1       0.18676
      forgets default 0.00000
Scenario 1 test Accuracy, 0.6340508806262231
Scenario 2 train Accuracy, 0.6910675381263617
                        value
skill param   class          
0     prior   default 0.00000
      learns  default 0.00195
      guesses 0       0.45104
              1       0.17377
      slips   0       0.02233
              1       0.07261
      forgets default 0.00000
1     prior   default 0.00000
      learns  default 0.00190
      guesses 0       0.45107
              1       0.17379
      slips   0       0.01213
              1       0.04022
      forgets default 0.00000
Scenario 2 test Accurac

In [11]:
for scenario in scenarios:
    if scenario["id"] > 6:
        scenario_id    = scenario["id"]
        num_tasks      = scenario["num_tasks"]
        difficulty_level = np.array(scenario["difficulty_level"])
        q_matrix       = np.array(scenario["q_matrix"])
        num_skills = scenario["num_skills"]
        train_df = load_training_data(scenario_id, "train")
        test_df = load_training_data(scenario_id, "test")

        model = Model(seed = 42, num_fits = 1)
        model.fit(data=train_df, defaults=defaults, multigs = True)#, forgets = True)
        # Perform evalution
        training_acc = model.evaluate(data=train_df, metric='accuracy')
        print(f"Scenario {scenario_id} train Accuracy, {training_acc}")
        print(model.params())
        test_acc = model.evaluate(data=test_df, metric='accuracy')
        print(f"Scenario {scenario_id} test Accuracy, {test_acc}")

        save_bkt(model, scenario_id, num_skills)

Process SpawnPoolWorker-1205:
Process SpawnPoolWorker-1204:
Process SpawnPoolWorker-1202:
Process SpawnPoolWorker-1203:
Process SpawnPoolWorker-1201:
Process SpawnPoolWorker-1210:
Process SpawnPoolWorker-1208:


KeyboardInterrupt: 